# imports

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import json
import matplotlib.pyplot as plt
import logging
logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)
from sklearn.metrics import mean_absolute_error, mean_squared_error
from feature_engine.datetime import DatetimeFeatures
from feature_engine.creation import CyclicalFeatures
from feature_engine.timeseries.forecasting import ExpandingWindowFeatures,LagFeatures
from sklearn.preprocessing import MinMaxScaler,StandardScaler
from sktime.transformations.series.fourier import FourierFeatures
from feature_engine.timeseries.forecasting import WindowFeatures
import holidays
from sklearn.ensemble import RandomForestRegressor
from mlforecast import MLForecast
from neuralforecast import NeuralForecast
import optuna
import itertools
import numpy as np
from xgboost import XGBRegressor
import re


c:\Users\CR58XM\AppData\Local\anaconda3\envs\nixtla_forecast\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
def plot_predictions(model_name, df_validation_y, all_preds_unscaled):
    plt.figure(figsize=(14, 7))

    # Plot actual values
    plt.plot(df_validation_y.index, df_validation_y.values, label='Actual Values', color='blue', linewidth=2)

    # Plot predicted values
    plt.plot(all_preds_unscaled.index, all_preds_unscaled.values, label=f'Predicted Values ({model_name})', color='red', linestyle='-', linewidth=2)

    # Add labels and title
    plt.xlabel('Time')
    plt.ylabel('Load')
    plt.title(f'Actual vs Predicted Values ({model_name})')

    # Add legend
    plt.legend()

    # Rotate x-ticks for better readability
    plt.xticks(rotation=45)

    # Show the plot
    plt.tight_layout()
    plt.show()

def CreateWorkHourFeature(input_data):
    """
    Receives as input a DataFrame or Series and outputs a DataFrame with the working hours during the day.
    When the day of the week is larger than 4, it is considered a weekend (1), otherwise, it's a workday (0).
    During workdays and between 8:00 and 17:00, it is considered a working hour.

    Parameters:
    input_data (DataFrame or Series): Input data with a DatetimeIndex.

    Returns:
    DataFrame: DataFrame with the added "WorkingHour_flag" column.
    """
    if isinstance(input_data, pd.Series):
        input_df = pd.DataFrame(input_data)
    elif isinstance(input_data, pd.DataFrame):
        input_df = input_data
    else:
        raise ValueError("Input must be a DataFrame or Series.")

    assert isinstance(input_df.index, pd.DatetimeIndex), "Index must be a datetime index."

    input_df["dayOfWeek"] = input_df.index.dayofweek
    input_df.loc[input_df["dayOfWeek"] > 4, "weekendFlag"] = 1
    input_df.loc[input_df["dayOfWeek"] < 5, "weekendFlag"] = 0
    input_df["hour"] = input_df.index.hour
    input_df["WorkingHour_flag"] = 0
    input_df.loc[((input_df["hour"] > 8) & (input_df["hour"] < 17) & (input_df["weekendFlag"] == 0)), "WorkingHour_flag"] = 1
    input_df.drop(["hour", "dayOfWeek", "weekendFlag"], axis=1, inplace=True)

    return input_df


def ListCreatorFlagger(df, substrings=['flag', 'cos', 'sin','day_of_week', 'day_of_month', 'weekend', 'days_in_month', 'hour', 'minute']):
    """
    A function that separates the columns containing specified substrings from those that don't.
    df is the dataframe in question and the substring is a list.
    """
    flag_columns = [col for col in df.columns if any(substring in col for substring in substrings)]

    if not flag_columns:
        print("No columns with the specified substrings found.")
        return None, None

    non_flag_columns = [col for col in df.columns if col not in flag_columns]

    return non_flag_columns, flag_columns


def HolidayFeatureCreator(input_data, countries):
    """
    Receives a DataFrame or Series and creates a column 'Holidays_flag'
    which is 1 if the date is a holiday in ANY of the given countries.
    """

    if isinstance(input_data, pd.Series):
        input_df = pd.DataFrame(input_data)
    elif isinstance(input_data, pd.DataFrame):
        input_df = input_data.copy()
    else:
        raise ValueError("Input must be a DataFrame or Series.")

    if not isinstance(input_df.index, pd.DatetimeIndex):
        raise ValueError("Index must be a DatetimeIndex.")

    years = input_df.index.year.unique().tolist()

    country_map = {
        "Germany": holidays.DE,
        "Ireland": holidays.IE,
        "Portugal": holidays.PT,
        "Denmark": holidays.DK,
    }

    holiday_dates = set()

    for country in countries:
        if country not in country_map:
            raise ValueError(f"{country} not supported")
        h = country_map[country](years=years)
        holiday_dates.update(h.keys())

    # convert index dates to pandas Index so .isin works
    index_dates = pd.Index(input_df.index.date)

    input_df["Holidays_flag"] = index_dates.isin(holiday_dates).astype(int)

    return input_df




def TimeRelatedFeatureConstructor(df):
  """
  Works only in a dataframe as input: run the other functions first.
  Extracts time-related features
  """
  TimeFeaturesToExtract=["day_of_week","weekend","hour",] #consider to add more
  dtfs=DatetimeFeatures(variables="index", features_to_extract=TimeFeaturesToExtract, drop_original=False)
  df=dtfs.fit_transform(df)

  CyclicalFeaturesToExtract=["day_of_week","hour",]
  cyclical_dtfs=CyclicalFeatures(variables=CyclicalFeaturesToExtract,drop_original=False)
  df=cyclical_dtfs.fit_transform(df)
  return df


def FourierFeatureConstructor(df, granularity, fourier_terms_list):
    # Extract numerical part of granularity
    number_part = ''.join(filter(str.isdigit, granularity))
    number_int = int(number_part) if number_part else 1  # Fallback to 1 to avoid division by zero

    # Calculate minutes per hour, ensuring no division by zero
    minutes4hour = 60 / number_int if number_int != 0 else 60

    # Define seasonal periods (sp_list) for Fourier transformation
    sp_list = [
        max(minutes4hour, 4),                 # Hourly - for 15min, this should be 4
        max(24 * minutes4hour, 96),           # Daily - for 15min, this should be 96
        max(24 * 7 * minutes4hour, 672),      # Weekly - for 15min, this should be 672
        max(24 * 30 * minutes4hour, 2880)     # Monthly - for 15min, this should be 2880
    ]

    # Fourier transformer setup
    Fourier_Transformer = FourierFeatures(
        sp_list=sp_list,
        fourier_terms_list=fourier_terms_list,
        freq=granularity,
        keep_original_columns=True
    )

    # Apply Fourier transformation
    Fourier_Transformer.fit(df)
    df = Fourier_Transformer.transform(df)
    return df



def WindowFeaturesConstructor(df, granularity, ListWithNoFlags):
    """
    This is a function that makes a list of 4 window features starting from double the granularity and following by doubling the previous value
    """
    number_part = ''.join(filter(str.isdigit, granularity))
    number_int = int(number_part)
    double_granularity = 2 * number_int
    time_intervals = [double_granularity]

    # Calculate subsequent values
    for i in range(3):
        time_intervals.append(time_intervals[-1] * 2)

    windowlist = [interval // number_int for interval in time_intervals]  # Corrected division
    functionsList = ["mean", "std"]
    WindownFeatureTransformer = WindowFeatures(variables=ListWithNoFlags,
                                               functions=functionsList,
                                               window=windowlist,
                                               freq=granularity,
                                               drop_original=False)

    df = WindownFeatureTransformer.fit_transform(df)
    return df

def ExpandingWindowFeatureConstructor(df,ListWithNoFlags):
  functionsList=["mean","std"]
  frequency = pd.infer_freq(df.index) #infer the frequency from the dataframe
  ExpandingWindownFeatureTransformer=ExpandingWindowFeatures(variables=ListWithNoFlags,
                                                           functions=functionsList,
                                                           freq=frequency, #I put the freq to shift it down! but now it is performed automatically!
                                                           drop_original=False)
  df=ExpandingWindownFeatureTransformer.fit_transform(df)
  return df

def WeightedLinearFeatureMaker(df,ListWithNoFlags,granularity):
  """
  This is a function that takes the original DF and modifies the continious value columns
  Inputs: Dataframe, List of columns that are continous values, daily window to slide, weights of the values
  """
  number_part = ''.join(filter(str.isdigit, granularity))
  Minutedensity=int(number_part)
  Window=int((60/Minutedensity)*24) #288 means a daily window
  weights=np.arange(1,Window+1)

  # if i had hourly data then i would have had np.arange(1,24*7) for a weekly window

  def weighted_mean (x,weights):
    return (weights*x).sum()/weights.sum()

  def weighted_std(x,weights):
    mean_w= weighted_mean(x, weights)
    var_w= (weights* (x-mean_w)**2).sum()/weights.sum()
    return np.sqrt(var_w)

  # LETS make the weighted mean column
  for i in ListWithNoFlags:
    result=(
        df[i]
        .rolling(window=Window) #here we pick a window size. Needs to be the same as the len(weights)
        .apply(weighted_mean, args=(weights,))
        .shift(1)#shift by 1 to avoid data leakage
        .to_frame()#convert series to df
        )

    result.columns=[str(i)+"_weighted_"+str(Window)+"_mean"]
    df=df.join(result)

  for i in ListWithNoFlags:
    result=(
        df[i]
        .rolling(window=Window) #here we pick a window size. Needs to be the same as the len(weights)
        .apply(weighted_std, args=(weights,))
        .shift(1)#shift by 1 to avoid data leakage
        .to_frame()#convert series to df
        )

    result.columns=[str(i)+"_weighted_"+str(Window)+"_std"]
    df=df.join(result)
  return df

def ExpWeightMeanMaker(df,ListWithNoFlags,granularity):
  """
  This is a function that makes exp weighted average with a sliding window approach
  """
  number_part = ''.join(filter(str.isdigit, granularity))
  Minutedensity=int(number_part)
  Window=int((60/Minutedensity)*24) #288 means a daily window

  def exp_weights(alpha,window_size):
    """
    a function to calculate the weights for every single component of our sliding windown
    """
    weights=np.ones(window_size) #initializing weights
    for ix in range(window_size):
      weights[ix]=(1-alpha)**(window_size-1-ix)
    return weights

  def exp_weighted_mean(x):
    """
    a functions that calculates the exp weigted mean
    """

    weights=exp_weights(alpha=0.05, window_size=len(x)) # HERE WE SET THE ALPHA
    return (weights*x).sum()/weights.sum()

  for i in ListWithNoFlags:
    result=(
        df[i]
        .rolling(window=int(Window))
        .agg([exp_weighted_mean])
        .shift(1)
    )


    result.columns=[str(i)+"_Exp_weighted_"+str(Window)+"_SL.win"]
    df=df.join(result)
  return df



def FeatureLagger(df, ListOfFeatures, granularity, PredictionHorizon):
    number_part = ''.join(filter(str.isdigit, granularity))
    Minutedensity = int(number_part)

    time_intervals = [f"{i * Minutedensity}min" for i in range(1, PredictionHorizon + 1)]

    lag_transformer = LagFeatures(
        variables=ListOfFeatures,
        freq=time_intervals,
        drop_original=False
    )

    df = lag_transformer.fit_transform(df)
    return df



def ErrorCalculator(name, y_true, y_pred):
    errors = {"Pipelines": name,
              "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
              "MAE": mean_absolute_error(y_true, y_pred),
              "MSE": mean_squared_error(y_true, y_pred),

             }
    return errors


def separate_future_past_features(df_columns):
    """
    Separates future and past features from a list of dataframe columns.

    Args:
        df_columns (list): A list of column names from the dataframe.

    Returns:
        dict: A dictionary with keys 'future_features' and 'past_features', containing the respective lists of column names.
    """
    future_keywords = ['sin', 'cos', 'weekend', 'hour', 'holiday', 'minute', 'day','+']

    future_features = []
    past_features = []

    for col in df_columns:
        # Check if the column contains "+" in its name to classify as a past feature
        if any(keyword in col.lower() for keyword in future_keywords):
            future_features.append(col)
        # Columns that don't meet the above conditions are considered past features by default
        else:
            past_features.append(col)

    return future_features, past_features


def plot_errors (ErrorSeries):
  """
  This is a function that plots the features that are not
  """
  import matplotlib as mpl
  import matplotlib.pyplot as plt
  import matplotlib.path as mpath
  import numpy as np

  import matplotlib.pyplot as plt
  import numpy as np

  x = np.arange(len(ErrorSeries.index))
  y = ErrorSeries.values
  labels = ErrorSeries.index

  plt.figure(1,figsize=(13,5))
  plt.style.use("seaborn-v0_8-whitegrid")
  plt.plot(x, y)

  plt.xticks(x, labels, rotation =40)
  plt.ylabel('RMSE [€/MWh]', wrap=True)
  plt.xlabel('Features', wrap=True)


  plt.margins(0.05)

  plt.subplots_adjust(bottom = 0.05)
  plt.show()


def select_features_minimum_plus_others(series):
    """
    Takes a pandas Series and selects the features that:
    - Include all features up to the minimum error.
    - After the minimum error, only include features that reduce the error compared to the previous one.
    - Ensures no duplicate features are added.

    Parameters:
    - series: A pandas Series where index are feature names and values are errors.

    Returns:
    - A list of selected feature names without duplicates.
    """
    # Find the index of the minimum value
    min_idx = series.idxmin()

    # Select all features up to and including the minimum
    selected_features = list(dict.fromkeys(series[:min_idx].index.tolist() + [min_idx]))

    # After the minimum, keep only the features that decrease the error
    after_min_series = series[min_idx:]

    # Loop through the series after the minimum value and add features that decrease the error
    for i in range(1, len(after_min_series)):
        if after_min_series[i] < after_min_series[i - 1]:
            feature = after_min_series.index[i]
            if feature not in selected_features:
                selected_features.append(feature)

    return selected_features

def keep_indices_till_min(series):
    """
    Keeps all index values from the series up to and including the minimum value using a for loop,
    while ensuring no duplicates are added.

    Parameters:
    - series: A pandas Series where the index are feature names and the values are errors.

    Returns:
    - A list of unique index values (features) up to and including the minimum error value.
    """
    # Initialize an empty list to store the selected indices
    selected_features = []

    # Find the minimum value in the series
    min_value = series.min()

    # Loop over the series
    for idx, value in series.items():
        # Add the current index to the selected features only if it's not already present
        if idx not in selected_features:
            selected_features.append(idx)

        # If the current value is the minimum, stop the loop
        if value == min_value:
            break

    return selected_features

def laggedColumnCreator(df,columnName,lagStart, lagInterval, lagEnd):
  for i in range(lagStart, lagEnd+1, lagInterval):
     newColumnName = columnName + "-" + str(i) + "step" #you gotta put it in string
     df[newColumnName] = df[columnName].shift(i)
  return df


def make_splits(
    test_start_str: str,
    freq: str = "15min",      # your timestep
    val_days: int = 14,
    train_steps: int = 7000,
    test_days: int = 14,       # length of test period
):
    step = pd.to_timedelta(freq)

    # TEST
    test_start = pd.Timestamp(test_start_str)
    # If you slice df.loc[start:end] (inclusive), use -step to get exactly `test_days` worth of data
    test_end = test_start + pd.Timedelta(days=test_days)

    # VALIDATION (ends one step before test_start)
    validation_end = test_start - step
    # `val_days` long, inclusive: end - start = val_days days - step
    validation_start = validation_end - pd.Timedelta(days=val_days) + step

    # TRAIN (ends one step before validation_start)
    train_end = validation_start - step
    # exactly `train_steps` steps long: end - start = (train_steps - 1) * step
    train_start = validation_start - train_steps * step

    return {
        "train_start": train_start,
        "train_end": train_end,
        "validation_start": validation_start,
        "validation_end": validation_end,
        "test_start": test_start,
        "test_end": test_end,
    }


def FeatureSelection(regressor, DF_features, DF_target, ordered_features_list, test_size=672, tolerance=400):
    """
    This function receives a regressor model, the features that the model was trained with,
    and the target that it had to forecast. Starting from the most important feature,
    we find the error of the TimeSeries Cross-Validation with a fixed test size.
    By adding features, we find the new error of the forecast.

    Parameters:
    - regressor: The regression model to use for training and prediction.
    - DF_features: DataFrame containing the features.
    - DF_target: Series containing the target variable.
    - ordered_features_list: List of features ordered by importance (e.g., from SHAP analysis).
    - test_size: Number of steps to use in the test set (default is 672).
    - tolerance: Number of features to add before stopping if no improvement in error (default is 20).

    Returns:
    - ErrorSeries: A pandas Series with the errors for each step of feature addition.
    """

    feature_list = []  # Empty list of features
    error_list = []  # Empty list to store errors for each set of features
    total_samples = len(DF_features)  # Total number of samples in the dataset
    n_splits = 5  # Number of splits (fixed)
    no_improvement_count = 0  # Count features added without improvement
    min_error = float('inf')  # Start with a large error to track the minimum error

    for i in ordered_features_list:
        # Start the loop with the best feature and append the next ones
        feature_list.append(i)

        X = DF_features[feature_list].to_numpy()
        y = DF_target.to_numpy()

        #print(f"Performing feature selection with features: {feature_list}")

        # Custom logic to create splits with a fixed test size of 672
        splits = []
        start_train_size = total_samples - (n_splits * test_size)  # Calculate where to start training

        for split in range(n_splits):
            train_end = start_train_size + split * test_size
            test_start = train_end
            test_end = test_start + test_size

            if test_end <= total_samples:  # Ensure the test set is within the bounds
                splits.append((list(range(0, train_end)), list(range(test_start, test_end))))

        TimeSeriesCVerror = []  # MSE errors for each fold

        # Time series cross-validation with fixed test size
        for train_index, test_index in splits:
            #print(f"TRAIN: {train_index}, TEST: {test_index}")
            X_train, X_test = X[train_index], X[test_index]
            y_train, y_test = y[train_index], y[test_index]

            # Train the regressor and predict
            regressor.fit(X_train, y_train)
            predicted_val = regressor.predict(X_test)

            # Calculate the error for this fold
            Error = np.sqrt(mean_squared_error(y_test, predicted_val))
            TimeSeriesCVerror.append(Error)
            #print(f"This is the error for one TS iteration: {Error}")

        # Calculate the average error across all splits
        TS_CV_error = sum(TimeSeriesCVerror) / len(TimeSeriesCVerror)
        #print(f"Cumulative error of the last steps: {TS_CV_error}")
        error_list.append(TS_CV_error)  # Store the error for this set of features

        # Check if the error improved
        if TS_CV_error < min_error:
            min_error = TS_CV_error  # Update the minimum error
            no_improvement_count = 0  # Reset the no-improvement count
        else:
            no_improvement_count += 1  # Increment if there's no improvement

        # Break the loop if no improvement is observed after 20 features
        if no_improvement_count >= tolerance:
            print(f"No improvement after {tolerance} features. Stopping early.")
            break

    # Create a pandas Series to store the error associated with each feature
    ErrorSeries = pd.Series(error_list, index=feature_list)

    # Plot the errors using a custom plot function
    plot_errors(ErrorSeries)

    return ErrorSeries



def build_feature_matrix(
    df_target,
    df_features,
    countries,
    granularity="15min",
    prediction_horizon=96,
    fourier_terms_list=[2, 2, 2, 2]
):
    """
    Combines target + exogenous features and creates engineered features.
    Returns:
        full_df: dataframe containing target and engineered features
        target_col: name of target column
    """

    target_name = df_target.name if df_target.name is not None else "target"

    full_df = pd.DataFrame(index=df_target.index)
    full_df[target_name] = df_target.astype(float)
    full_df = full_df.join(df_features.astype(float), how="left")

    # Calendar / deterministic features
    full_df = CreateWorkHourFeature(full_df)
    full_df = HolidayFeatureCreator(full_df, countries=countries)
    full_df = TimeRelatedFeatureConstructor(full_df)
    full_df = FourierFeatureConstructor(
        full_df,
        granularity=granularity,
        fourier_terms_list=fourier_terms_list
    )

    # Separate columns for rolling/expanding transforms
    non_flag_cols, flag_cols = ListCreatorFlagger(full_df)

    # Remove target from the list if present
    non_flag_cols = [c for c in non_flag_cols if c != target_name]

    # Rolling/expanding on continuous predictors
    if len(non_flag_cols) > 0:
        full_df = WindowFeaturesConstructor(full_df, granularity, non_flag_cols)
        full_df = ExpandingWindowFeatureConstructor(full_df, non_flag_cols)
        full_df = WeightedLinearFeatureMaker(full_df, non_flag_cols, granularity)
        full_df = ExpWeightMeanMaker(full_df, non_flag_cols, granularity)

    lag_features = [target_name] + list(df_features.columns)

    # Lags of target
    full_df = FeatureLagger(
        full_df,
        ListOfFeatures=lag_features,
        granularity=granularity,
        PredictionHorizon=prediction_horizon
    )

    return full_df, target_name


def make_train_val_nextday_split(full_df, target_col, freq="15min", val_days=3, pred_len=96):
    step = pd.to_timedelta(freq)
    val_steps = int(pd.Timedelta(days=val_days) / step)

    full_df = full_df.dropna().copy()

    min_required = val_steps + pred_len
    if len(full_df) <= min_required:
        raise ValueError(
            f"Not enough usable history after dropna. "
            f"Need more than {min_required} rows, got {len(full_df)}."
        )

    train_df = full_df.iloc[:-val_steps].copy()
    val_df = full_df.iloc[-val_steps:].copy()

    X_train = train_df.drop(columns=[target_col])
    y_train = train_df[target_col]

    X_val = val_df.drop(columns=[target_col])
    y_val = val_df[target_col]

    last_ts = full_df.index[-1]
    future_index = pd.date_range(
        start=last_ts + step,
        periods=pred_len,
        freq=freq
    )

    return X_train, y_train, X_val, y_val, future_index, full_df
    
def FeatureSelectionOnValidation(regressor, X_train, y_train, X_val, y_val, ordered_features_list, tolerance=20):
    feature_list = []
    error_list = []

    no_improvement_count = 0
    min_error = np.inf

    for feat in ordered_features_list:
        feature_list.append(feat)

        regressor.fit(X_train[feature_list], y_train)
        pred_val = regressor.predict(X_val[feature_list])
        mse = mean_squared_error(y_val, pred_val)
        error_list.append(mse)

        if mse < min_error:
            min_error = mse
            no_improvement_count = 0
        else:
            no_improvement_count += 1

        if no_improvement_count >= tolerance:
            print(f"No improvement after {tolerance} added features. Stopping early.")
            break

    error_series = pd.Series(error_list, index=feature_list)
    return error_series


def tune_xgboost_on_validation(X_train, y_train, X_val, y_val):
    param_grid = {
        "n_estimators": [200, 400],
        "max_depth": [4, 6, 8],
        "learning_rate": [0.03, 0.05, 0.1],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
        "reg_lambda": [1, 5],
    }

    keys = list(param_grid.keys())
    best_model = None
    best_params = None
    best_mse = np.inf

    for values in itertools.product(*(param_grid[k] for k in keys)):
        params = dict(zip(keys, values))

        model = XGBRegressor(
            objective="reg:squarederror",
            random_state=42,
            tree_method="hist",
            **params
        )

        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        mse = mean_squared_error(y_val, y_pred)

        if mse < best_mse:
            best_mse = mse
            best_model = model
            best_params = params

    return best_model, best_params, best_mse

def build_future_feature_matrix(
    history_target,
    history_features,
    future_features,
    countries,
    freq="15min",
    pred_len=96,
    fourier_terms_list=[2, 2, 2, 2]
):
    """
    Builds a feature matrix for the next day.
    Assumes future_features is available for the future timestamps if you want weather.
    """

    future_index = future_features.index
    target_name = history_target.name if history_target.name is not None else "target"

    # combine history + empty future target
    df_hist = pd.DataFrame(index=history_target.index)
    df_hist[target_name] = history_target.astype(float)
    df_hist = df_hist.join(history_features, how="left")

    df_future = pd.DataFrame(index=future_index)
    df_future[target_name] = np.nan
    df_future = df_future.join(future_features, how="left")

    full_df = pd.concat([df_hist, df_future], axis=0)

    # deterministic features
    full_df = CreateWorkHourFeature(full_df)
    full_df = HolidayFeatureCreator(full_df, countries=countries)
    full_df = TimeRelatedFeatureConstructor(full_df)
    full_df = FourierFeatureConstructor(
        full_df,
        granularity=freq,
        fourier_terms_list=fourier_terms_list
    )

    # rolling features on exogenous continuous cols
    non_flag_cols, flag_cols = ListCreatorFlagger(full_df)
    non_flag_cols = [c for c in non_flag_cols if c != target_name]

    if len(non_flag_cols) > 0:
        full_df = WindowFeaturesConstructor(full_df, freq, non_flag_cols)
        full_df = ExpandingWindowFeatureConstructor(full_df, non_flag_cols)
        full_df = WeightedLinearFeatureMaker(full_df, non_flag_cols, freq)
        full_df = ExpWeightMeanMaker(full_df, non_flag_cols, freq)

    # lag target using history only
    full_df = FeatureLagger(
        full_df,
        ListOfFeatures=[target_name],
        granularity=freq,
        PredictionHorizon=pred_len
    )

    return full_df.loc[future_index].copy()

# with permutation

In [3]:
import os
import json
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean, RollingStd, ExpandingMean, ExpandingStd


# ============================================================
# CONFIG
# ============================================================
DAYS_JSON = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\dataset_days.json"
DATA_DIR  = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\DataCleaning\clean"
OUT_DIR   = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs"
SELECTED_FEATURES_DIR = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\Selected_features"



countries = ["Germany", "Ireland", "Portugal"]

days = ["day1", "day2", "day3", "day4", "day5"]

features = [
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "precipitation",
    "direct_radiation",
]

PRED_LEN = 96
VAL_DAYS = 3
CONTEXT_TAIL = 10000
FEATURE_IMPORTANCE_THRESHOLD = 0.90
PERM_N_REPEATS = 5

# Fixed RF selector parameters
RF_SELECTOR_PARAMS = {
    "n_estimators": 500,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 5,
    "max_features": "sqrt",
    "bootstrap": True,
    "random_state": 42,
    "n_jobs": -1,
}

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(SELECTED_FEATURES_DIR, exist_ok=True)


# ============================================================
# HELPER FUNCTIONS
# ============================================================
def make_long_target_exog(df_target, df_features, household):
    out = pd.DataFrame({
        "unique_id": household,
        "ds": df_target.index,
        "y": df_target.values,
    })
    out = out.merge(
        df_features.reset_index().rename(columns={df_features.index.name or "index": "ds"}),
        on="ds",
        how="left"
    )
    return out


def add_calendar_and_fourier_exog(df_long, countries, granularity="15min", fourier_terms_list=[2, 2, 2, 2]):
    """
    Adds only deterministic / exogenous features.
    No manual target lags here.
    Assumes the custom feature engineering functions are already in memory.
    """
    tmp = df_long.set_index("ds").copy()

    tmp = CreateWorkHourFeature(tmp)
    tmp = HolidayFeatureCreator(tmp, countries=countries)
    tmp = TimeRelatedFeatureConstructor(tmp)
    tmp = FourierFeatureConstructor(
        tmp,
        granularity=granularity,
        fourier_terms_list=fourier_terms_list
    )

    exog_cols = [c for c in features if c in tmp.columns]
    if len(exog_cols) > 0:
        tmp = WindowFeaturesConstructor(tmp, granularity, exog_cols)
        tmp = ExpandingWindowFeatureConstructor(tmp, exog_cols)
        tmp = WeightedLinearFeatureMaker(tmp, exog_cols, granularity)
        tmp = ExpWeightMeanMaker(tmp, exog_cols, granularity)

    tmp = tmp.reset_index()
    return tmp


def split_train_val_long(df_long, val_days=3, freq="15min"):
    step = pd.to_timedelta(freq)
    val_steps = int(pd.Timedelta(days=val_days) / step)

    df_long = df_long.sort_values("ds").reset_index(drop=True)

    if len(df_long) <= val_steps:
        raise ValueError(f"Not enough rows. Need > {val_steps}, got {len(df_long)}")

    train_df = df_long.iloc[:-val_steps].copy()
    val_df = df_long.iloc[-val_steps:].copy()

    return train_df, val_df


def build_mlf_definition(pred_len):
    """
    Build MLForecast object with a placeholder RF model.
    Used only for preprocessing / feature generation.
    """
    fcst = MLForecast(
        models={
            "rf": RandomForestRegressor(
                n_estimators=10,
                random_state=42,
                n_jobs=-1
            )
        },
        freq="15min",
        lags=list(range(1, pred_len + 1)),
        lag_transforms={
            1: [ExpandingMean(), ExpandingStd()],
            4: [RollingMean(window_size=4), RollingStd(window_size=4)],
            96: [RollingMean(window_size=96), RollingStd(window_size=96)],
        },
        date_features=[],
    )
    return fcst


def get_full_matrix_from_fcst(fcst, full_df_long):
    """
    Use MLForecast preprocessing to generate the full matrix,
    then we can split rows into train/validation after lag creation.
    """
    prep = fcst.preprocess(
        full_df_long,
        id_col="unique_id",
        time_col="ds",
        target_col="y",
        static_features=[],
        dropna=True
    )

    feature_cols = fcst.ts.features_order_
    X_full = prep[feature_cols].copy()
    y_full = prep["y"].copy()

    return prep, X_full, y_full, feature_cols


def select_features_with_permutation(
    X_train,
    y_train,
    X_val,
    y_val,
    selector_params,
    threshold=0.90,
    n_repeats=5
):
    """
    1. Fit RF on train
    2. Compute permutation importance on validation
    3. Select features using cumulative importance threshold
    """
    selector = RandomForestRegressor(**selector_params)
    selector.fit(X_train, y_train)

    perm = permutation_importance(
        selector,
        X_val,
        y_val,
        n_repeats=n_repeats,
        random_state=42,
        n_jobs=-1,
        scoring="neg_root_mean_squared_error"
    )

    feature_importances = pd.Series(
        perm.importances_mean,
        index=X_val.columns
    ).sort_values(ascending=False)

    # clip negative importances to zero for cumulative selection
    feature_importances = feature_importances.clip(lower=0)

    # fallback in case everything becomes zero
    if feature_importances.sum() == 0:
        selected_features = feature_importances.index[:1].tolist()
    else:
        normalized_importance = feature_importances / feature_importances.sum()
        cumulative_importance = normalized_importance.cumsum()

        selected_features = cumulative_importance[cumulative_importance <= threshold].index.tolist()

        if len(selected_features) < len(feature_importances):
            selected_features.append(feature_importances.index[len(selected_features)])

    return selected_features, feature_importances, selector


def save_selected_features_report(
    save_path,
    feature_importances,
    selected_features,
    household,
    country,
    day
):
    report = pd.DataFrame({
        "feature": feature_importances.index,
        "importance": feature_importances.values,
    })

    total_imp = report["importance"].clip(lower=0).sum()
    if total_imp > 0:
        report["normalized_importance"] = report["importance"].clip(lower=0) / total_imp
        report["cumulative_importance"] = report["normalized_importance"].cumsum()
    else:
        report["normalized_importance"] = 0.0
        report["cumulative_importance"] = 0.0

    report["selected"] = report["feature"].isin(selected_features).astype(int)
    report["household"] = household
    report["country"] = country
    report["day"] = day
    report.to_csv(save_path, index=False)
    return report


# ============================================================
# MAIN
# ============================================================
with open(DAYS_JSON, "r") as f:
    dataset_days = json.load(f)

all_feature_selection_rows = []

for country in countries:
    print(f"\nProcessing country: {country}")

    data_path = os.path.join(DATA_DIR, f"dataset_{country.capitalize()}.csv")
    df = pd.read_csv(data_path, index_col="timestamp", parse_dates=True).sort_index()

    households = [c for c in df.columns if c not in features]

    for day in days:
        print(f"\n  Day: {day}")
        cutoff = pd.to_datetime(dataset_days[country][day])

        for household in households:
            print(f"\n    Household: {household}")

            # ------------------------------------------------
            # 1. Build history up to cutoff
            # ------------------------------------------------
            df_target = df.loc[df.index < cutoff, household].astype(float).dropna()

            if len(df_target) < 500:
                print(f"    {household} skipped: too little history")
                continue

            df_target = df_target.tail(CONTEXT_TAIL)
            df_features_hist = df.loc[df_target.index, features].copy()

            # ------------------------------------------------
            # 2. Build long dataframe with exogenous features
            # ------------------------------------------------
            long_df = make_long_target_exog(df_target, df_features_hist, household)
            long_df = add_calendar_and_fourier_exog(
                long_df,
                countries=[country],
                granularity="15min",
                fourier_terms_list=[2, 2, 2, 2]
            )
            long_df = long_df.dropna().copy()

            print("    usable rows before MLForecast lagging:", len(long_df))

            try:
                train_df_raw, val_df_raw = split_train_val_long(
                    long_df,
                    val_days=VAL_DAYS,
                    freq="15min"
                )
            except ValueError as e:
                print(f"    {household} skipped: {e}")
                continue

            val_start = val_df_raw["ds"].min()

            # ------------------------------------------------
            # 3. MLForecast preprocessing definition
            # ------------------------------------------------
            fcst_selector = build_mlf_definition(PRED_LEN)

            # ------------------------------------------------
            # 4. Generate FULL matrix from MLForecast
            #    Then split into train/val AFTER lag creation
            # ------------------------------------------------
            prep_full, X_full, y_full, feature_cols = get_full_matrix_from_fcst(
                fcst_selector,
                long_df
            )

            prep_full = prep_full.sort_values("ds").reset_index(drop=True)

            train_mask = prep_full["ds"] < val_start
            val_mask = prep_full["ds"] >= val_start

            X_train_full = prep_full.loc[train_mask, feature_cols].copy()
            y_train_full = prep_full.loc[train_mask, "y"].copy()

            X_val_full = prep_full.loc[val_mask, feature_cols].copy()
            y_val_full = prep_full.loc[val_mask, "y"].copy()

            if len(X_train_full) == 0 or len(X_val_full) == 0:
                print(f"    {household} skipped: empty train/validation matrix after MLForecast preprocessing")
                continue

            print("    Number of generated MLForecast features:", len(feature_cols))
            print("    Train rows after lagging:", len(X_train_full))
            print("    Val rows after lagging:", len(X_val_full))

            # ------------------------------------------------
            # 5. RF selector + permutation importance on validation
            # ------------------------------------------------
            selected_features, feature_importances, rf_selector = select_features_with_permutation(
                X_train=X_train_full,
                y_train=y_train_full,
                X_val=X_val_full,
                y_val=y_val_full,
                selector_params=RF_SELECTOR_PARAMS,
                threshold=FEATURE_IMPORTANCE_THRESHOLD,
                n_repeats=PERM_N_REPEATS
            )

            print("    Total generated features:", len(feature_importances))
            print("    Selected features:", len(selected_features))
            print("    Top 20 permutation importances:")
            print(feature_importances.head(20))

            # ------------------------------------------------
            # 6. Save selected features report
            # ------------------------------------------------
            selected_features_path = os.path.join(
                SELECTED_FEATURES_DIR,
                f"selected_features_perm_{country}_{day}_{household}.csv"
            )

            report_df = save_selected_features_report(
                save_path=selected_features_path,
                feature_importances=feature_importances,
                selected_features=selected_features,
                household=household,
                country=country,
                day=day
            )

            print(f"    Saved selected features to: {selected_features_path}")

            all_feature_selection_rows.append({
                "country": country,
                "day": day,
                "household": household,
                "n_total_features": len(feature_importances),
                "n_selected_features": len(selected_features),
                "top_feature_1": feature_importances.index[0] if len(feature_importances) > 0 else None,
                "top_feature_2": feature_importances.index[1] if len(feature_importances) > 1 else None,
                "top_feature_3": feature_importances.index[2] if len(feature_importances) > 2 else None,
                "selected_features_path": selected_features_path,
            })

# ============================================================
# SAVE SUMMARY
# ============================================================
if len(all_feature_selection_rows) > 0:
    fs_df = pd.DataFrame(all_feature_selection_rows)
    fs_path = os.path.join(OUT_DIR, "feature_selection_summary_permutation.csv")
    fs_df.to_csv(fs_path, index=False)
    print(f"\nSaved feature selection summary to: {fs_path}")


Processing country: Germany

  Day: day1

    Household: home_1
    usable rows before MLForecast lagging: 9904
    Number of generated MLForecast features: 197
    Train rows after lagging: 9425
    Val rows after lagging: 288


c:\Users\CR58XM\AppData\Local\anaconda3\envs\nixtla_forecast\Lib\site-packages\utilsforecast\processing.py:124: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[names] = values
c:\Users\CR58XM\AppData\Local\anaconda3\envs\nixtla_forecast\Lib\site-packages\utilsforecast\processing.py:124: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[names] = values
c:\Users\CR58XM\AppData\Local\anaconda3\envs\nixtla_forecast\Lib\site-packages\utilsforecast\processing.py:124: PerformanceWarning: DataFrame is highly fragmented.  This is usually

    Total generated features: 197
    Selected features: 48
    Top 20 permutation importances:
lag1                               44.871370
sin_2880.0_2                       15.926234
lag3                               15.575445
hour_cos                           12.781306
hour_sin                            9.145201
rolling_mean_lag4_window_size4      7.657793
lag4                                6.982465
lag2                                6.102316
rolling_std_lag4_window_size4       5.102717
lag96                               4.950612
cos_2880.0_2                        4.899441
hour                                4.263007
direct_radiation_window_16_mean     3.817562
lag85                               3.447605
direct_radiation_window_16_std      3.356675
direct_radiation_window_8_std       2.129968
sin_96.0_1                          1.928227
lag8                                1.811117
lag6                                1.727848
lag7                                1.616624
dtyp

c:\Users\CR58XM\AppData\Local\anaconda3\envs\nixtla_forecast\Lib\site-packages\utilsforecast\processing.py:124: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[names] = values
c:\Users\CR58XM\AppData\Local\anaconda3\envs\nixtla_forecast\Lib\site-packages\utilsforecast\processing.py:124: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[names] = values
c:\Users\CR58XM\AppData\Local\anaconda3\envs\nixtla_forecast\Lib\site-packages\utilsforecast\processing.py:124: PerformanceWarning: DataFrame is highly fragmented.  This is usually

    Total generated features: 197
    Selected features: 18
    Top 20 permutation importances:
lag3                                  89.352665
lag2                                  22.950418
lag14                                  9.847787
lag1                                   2.340193
lag6                                   1.967987
lag89                                  1.262912
temperature_2m                         1.187080
lag28                                  1.151491
hour_sin                               0.771720
lag44                                  0.669031
temperature_2m_window_2_mean           0.603589
wind_speed_10m_window_8_mean           0.555941
lag18                                  0.542278
relative_humidity_2m_window_8_mean     0.507219
wind_speed_10m                         0.497213
temperature_2m_window_4_mean           0.468736
lag17                                  0.454778
wind_speed_10m_window_16_mean          0.453258
temperature_2m_window_8_mean           0

...

    Total generated features: 197
    Selected features: 44
    Top 20 permutation importances:
lag1                                       29.753287
hour_cos                                    1.582861
lag2                                        1.478952
cos_2880.0_2                                1.354194
hour                                        1.282683
sin_2880.0_2                                1.123542
lag95                                       0.967395
direct_radiation_window_16_std              0.869544
direct_radiation_window_8_mean              0.785722
direct_radiation_window_16_mean             0.735902
direct_radiation_window_4_mean              0.600812
hour_sin                                    0.598862
direct_radiation                            0.579878
lag96                                       0.565851
direct_radiation_window_2_mean              0.547761
lag17                                       0.533755
direct_radiation_Exp_weighted_96_SL.win     0.527906
lag

c:\Users\CR58XM\AppData\Local\anaconda3\envs\nixtla_forecast\Lib\site-packages\utilsforecast\processing.py:124: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[names] = values
c:\Users\CR58XM\AppData\Local\anaconda3\envs\nixtla_forecast\Lib\site-packages\utilsforecast\processing.py:124: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[names] = values
c:\Users\CR58XM\AppData\Local\anaconda3\envs\nixtla_forecast\Lib\site-packages\utilsforecast\processing.py:124: PerformanceWarning: DataFrame is highly fragmented.  This is usually

    Total generated features: 197
    Selected features: 35
    Top 20 permutation importances:
lag1                                           12.771497
relative_humidity_2m_Exp_weighted_96_SL.win     1.002867
lag2                                            0.909313
hour                                            0.776276
rolling_std_lag4_window_size4                   0.590312
cos_2880.0_2                                    0.564877
direct_radiation_window_4_mean                  0.436881
relative_humidity_2m_window_2_mean              0.375758
sin_2880.0_2                                    0.363805
relative_humidity_2m_window_16_mean             0.330395
direct_radiation_window_8_std                   0.329873
relative_humidity_2m_window_4_mean              0.324084
direct_radiation_window_2_std                   0.300072
relative_humidity_2m_window_8_mean              0.292668
relative_humidity_2m                            0.282652
direct_radiation_weighted_96_std                0

c:\Users\CR58XM\AppData\Local\anaconda3\envs\nixtla_forecast\Lib\site-packages\utilsforecast\processing.py:124: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[names] = values
c:\Users\CR58XM\AppData\Local\anaconda3\envs\nixtla_forecast\Lib\site-packages\utilsforecast\processing.py:124: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[names] = values
c:\Users\CR58XM\AppData\Local\anaconda3\envs\nixtla_forecast\Lib\site-packages\utilsforecast\processing.py:124: PerformanceWarning: DataFrame is highly fragmented.  This is usually

    Total generated features: 197
    Selected features: 22
    Top 20 permutation importances:
lag1                                       38.533679
lag2                                        5.040445
direct_radiation_window_16_mean             2.016178
lag3                                        1.667515
temperature_2m_window_16_std                1.606136
hour                                        1.378852
direct_radiation_window_16_std              1.347598
temperature_2m_window_8_std                 1.053827
cos_2880.0_2                                1.019665
sin_2880.0_2                                0.876213
direct_radiation_Exp_weighted_96_SL.win     0.649822
lag96                                       0.615965
lag4                                        0.582667
direct_radiation_window_4_mean              0.529469
direct_radiation_window_2_mean              0.482935
lag48                                       0.450917
cos_2880.0_1                                0.421322
lag

# end